# Análise de Operações Financeiras no Brasil (2011–2023)

## Contexto

Este projeto analisa a evolução das operações financeiras realizadas no Brasil entre 2011 e 2023, segmentadas por canal de atendimento:

- **Agências e Postos de Atendimento** — atendimento presencial tradicional
- **ATM** — caixas eletrônicos
- **Internet Banking** — operações via navegador web
- **Tel Celular** — operações via aplicativo móvel

## Fonte dos Dados

Os dados foram obtidos do Banco Central do Brasil e disponibilizados publicamente via Google Sheets. Cada linha representa um tipo de operação (ex: transferências, pagamentos, saques) dentro de um canal específico, com os valores anuais distribuídos em colunas.

## Objetivos da Análise

1. Visualizar a **tendência geral** do volume de operações por canal ao longo do tempo
2. Comparar a **distribuição dos tipos de operação** dentro de cada canal
3. Calcular a **taxa de crescimento anual** de cada canal
4. Analisar individualmente a evolução de cada canal

In [ ]:
import pandas as pd
import io
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Carregamento e Preparação dos Dados

Os dados são carregados diretamente da fonte pública e passam por um processo de **ETL** (Extract, Transform, Load):

- **Extração:** leitura do CSV via URL
- **Transformação:** os anos estão originalmente em colunas separadas (formato wide). Usamos `melt()` para converter para o formato long — uma linha por combinação de canal, operação e ano — o que facilita a análise e a visualização com seaborn
- **Limpeza:** os valores estão formatados com separador de milhar (`.`), o que impede a conversão direta para numérico. Removemos o separador antes de converter

In [ ]:
url = 'https://docs.google.com/spreadsheets/d/1EUTvdJUDptvnkJknB5sl0A7a-PgvpHA0wYx_FKXxK0M/export?format=csv'
response = requests.get(url)
opfin = pd.read_csv(io.StringIO(response.text))
opfin.head()

In [ ]:
# Visão geral do dataset
print('Shape:', opfin.shape)
print()
print('Canais disponíveis:')
print(opfin['Canal'].unique())
print()
print('Tipos de operação disponíveis:')
print(opfin['Operacao'].unique())
print()
print('Anos disponíveis (colunas):')
print([c for c in opfin.columns if c not in ['Canal', 'Operacao']])
print()
opfin.head(10)


In [ ]:
# Transforma o formato wide (anos em colunas) para o formato long (uma linha por ano)
# Isso é necessário para que seaborn consiga plotar séries temporais por canal
df_melted = opfin.melt(id_vars=['Canal', 'Operacao'],
                       var_name='Ano',
                       value_name='Valor')

# Converte a coluna 'Ano' de string para inteiro
df_melted['Ano'] = df_melted['Ano'].astype(int)

# Remove o separador de milhar '.' caso o valor seja string, depois converte para numérico
# errors='coerce' transforma valores inválidos em NaN em vez de lançar erro
if df_melted['Valor'].dtype == object:
    df_melted['Valor'] = df_melted['Valor'].str.replace('.', '', regex=False)
df_melted['Valor'] = pd.to_numeric(df_melted['Valor'], errors='coerce')

# Agrega por canal e ano para obter o volume total de operações por canal a cada ano
df_trend = df_melted.groupby(['Canal', 'Ano'])['Valor'].sum().reset_index()

print(f'Total de registros após melt: {len(df_melted)}')
print(f'Valores nulos em Valor: {df_melted["Valor"].isna().sum()}')
df_melted.head()


## 1. Tendência Geral por Canal (2011–2023)

O gráfico abaixo mostra a evolução do volume total de operações para cada canal ao longo do período analisado. Ele permite identificar quais canais cresceram, estagnaram ou perderam relevância.

In [ ]:
plt.figure(figsize=(12, 8))
sns.lineplot(data=df_trend, x='Ano', y='Valor', hue='Canal', marker='o')

plt.title('Tendência das Operações Financeiras por Canal (2011–2023)')
plt.xlabel('Ano')
plt.ylabel('Volume Total de Operações')
plt.grid(True)
plt.xticks(df_trend['Ano'].unique(), rotation=45)
plt.legend(title='Canal')
plt.tight_layout()
plt.show()

## 2. Distribuição de Operações por Tipo em Cada Canal

Aqui comparamos quais tipos de operação (transferências, pagamentos, saques, etc.) predominam em cada canal. Isso revela as especialidades e os perfis de uso de cada meio de atendimento.

In [ ]:
# Agrega por canal e tipo de operação, somando todos os anos
df_ops_by_channel = df_melted.groupby(['Canal', 'Operacao'])['Valor'].sum().reset_index()

# catplot cria automaticamente um subplot por canal (col='Canal')
g = sns.catplot(data=df_ops_by_channel, x='Operacao', y='Valor', col='Canal',
                col_wrap=2, kind='bar', height=4, aspect=1.8, sharey=True)
g.set_axis_labels("Tipo de Operação", "Valor Total das Operações")
g.set_titles("Canal: {col_name}")
g.set_xticklabels(rotation=45, ha='right')
plt.suptitle('Distribuição Total das Operações Financeiras por Tipo em Cada Canal (2011–2023)', y=1.02)
plt.tight_layout()
plt.show()

## 3. Taxa de Crescimento Anual por Canal

A taxa de crescimento anual indica a variação percentual do volume de operações de um ano para o outro. Valores positivos indicam crescimento, negativos indicam retração. A linha tracejada em 0% serve como referência visual.

In [ ]:
# Ordena por canal e ano para garantir que pct_change() calcule corretamente
df_growth = df_trend.sort_values(by=['Canal', 'Ano'])

# pct_change() calcula a variação percentual em relação ao valor do período anterior
df_growth['Crescimento_Anual'] = df_growth.groupby('Canal')['Valor'].pct_change() * 100

plt.figure(figsize=(12, 8))
sns.lineplot(data=df_growth, x='Ano', y='Crescimento_Anual', hue='Canal', marker='o')
plt.title('Taxa de Crescimento Anual das Operações Financeiras por Canal')
plt.xlabel('Ano')
plt.ylabel('Crescimento Anual (%)')
plt.grid(True)
plt.xticks(df_growth['Ano'].unique(), rotation=45)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.legend(title='Canal')
plt.tight_layout()
plt.show()

## 4. Análise Individual por Canal

Por fim, analisamos a evolução de cada canal separadamente, detalhando o volume por tipo de operação ao longo dos anos. Essa visão granular permite identificar quais operações específicas impulsionaram o crescimento ou a queda de cada canal.

In [ ]:
# Dicionário mapeando o nome do canal (como aparece nos dados) ao seu rótulo de exibição
canais = {
    'Agencias e Postos de Atendimento': 'Agências e Postos de Atendimento',
    'ATM': 'ATM (Caixas Eletrônicos)',
    'Internet Banking': 'Internet Banking',
    'Tel Celular': 'Telefone Celular (App)'
}

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
axs = axs.flatten()

for idx, (canal_key, canal_label) in enumerate(canais.items()):
    # Filtra apenas as linhas do canal atual
    df_canal = df_melted[df_melted['Canal'] == canal_key]

    sns.lineplot(
        data=df_canal,
        x='Ano',
        y='Valor',
        hue='Operacao',
        marker='o',
        ax=axs[idx]
    )

    axs[idx].set_title(canal_label, fontsize=12)
    axs[idx].set_xlabel('Ano')
    axs[idx].set_ylabel('Volume de Operações')
    axs[idx].tick_params(axis='x', rotation=45)
    axs[idx].grid(True)
    axs[idx].legend(title='Operação', fontsize=7, title_fontsize=8)

plt.suptitle('Evolução das Operações por Tipo em Cada Canal (2011–2023)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Conclusões

A análise dos dados do Banco Central revela transformações significativas no sistema financeiro brasileiro entre 2011 e 2023:


In [ ]:
# Resumo quantitativo: variação total por canal no período
resumo = df_trend.groupby('Canal').apply(
    lambda x: x.sort_values('Ano').assign(
        Variacao_Total=lambda df: (df['Valor'].iloc[-1] - df['Valor'].iloc[0]) / df['Valor'].iloc[0] * 100
    )[['Canal', 'Variacao_Total']].iloc[-1]
).reset_index(drop=True)

print('Variação total do volume de operações por canal (2011–2023):')
print()
for _, row in resumo.iterrows():
    sinal = '+' if row['Variacao_Total'] > 0 else ''
    print(f"  {row['Canal']:<40} {sinal}{row['Variacao_Total']:.1f}%")

print()
print('Observações:')
print('  • O canal Telefone Celular apresentou o maior crescimento do período,')
print('    refletindo a aceleração da digitalização financeira no Brasil.')
print('  • Agências e ATMs mostram tendência de queda, indicando migração')
print('    dos usuários para canais digitais.')
print('  • O Internet Banking cresceu de forma consistente até ~2018,')
print('    quando o celular passou a dominar.')
